# Part 2: Probability Distributions in Machine Learning
**⏱ This section takes approximately 30 minutes.**

## Scenario: Understanding Your Listeners' Behaviour

You're a data scientist at a music streaming platform. Your manager asks:
> "What does our listening session data look like? Are most people listening for similar amounts of time, or is it all over the place?"

The answer depends on the **shape** of your data — its *distribution*.

Understanding distributions is critical in ML because:
- Your model assumptions often depend on the distribution shape
- Knowing the distribution helps you spot anomalies and outliers
- The Central Limit Theorem (coming up!) is why so many ML techniques work

**By the end of this section you'll be able to:**
- Identify and generate uniform and normal distributions
- Explain why the Central Limit Theorem matters for ML
- Connect distribution shapes to real business decisions

In [ ]:
import numpy as np
import scipy.stats as st
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
np.random.seed(42)
sns.set_style('whitegrid')
print("✅ Libraries loaded!")

## 🎲 The Uniform Distribution — "Everything is equally likely"

**In plain English:** A uniform distribution means every outcome has exactly the same probability of occurring.

**Streaming analogy:** Imagine you randomly assign each new user to one of 10 A/B test groups. Each group has an equal 10% chance of being selected. That's a uniform distribution.

**When you'll see it in ML:**
- Random initialisation of weights in neural networks
- Hyperparameter search (trying values equally across a range)
- Random sampling when you have no prior beliefs

Let's generate one and visualise it.

### ⏸️ Pause and Predict

We're about to generate 10,000 values from a uniform distribution between 0 and 1, then plot a histogram.

**What shape do you expect the histogram to be?**
- A tall spike in the middle?
- A bell curve?
- A flat rectangle?

*Write your prediction here:*

In [ ]:
# Generate 10,000 values — each equally likely to be anywhere from 0 to 1
# This simulates: "pick a random number between 0 and 1"
session_group = np.random.uniform(size=10_000)

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(session_group, bins=40, color='steelblue', edgecolor='white')
ax.set_xlabel('Value', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Uniform Distribution — all values equally likely', fontsize=13)
plt.tight_layout()
plt.show()

print(f"Mean:  {np.mean(session_group):.3f}  (expected: 0.500)")
print(f"Min:   {np.min(session_group):.3f}")
print(f"Max:   {np.max(session_group):.3f}")

### 💡 What do you notice?

The histogram is **flat** — a rectangle. Every value between 0 and 1 is equally likely, so all bars are roughly the same height.

This is very different from how most real-world data looks. Most measurements in nature and business cluster around a central value — which brings us to the normal distribution.

## 🔔 The Normal (Gaussian) Distribution — "Most things cluster in the middle"

**In plain English:** Most values cluster around the average. Values become progressively rarer as you move further from the centre in either direction. This creates the famous "bell curve" shape.

**Streaming analogy:** Most users listen to songs for 2–4 minutes. Very few skip after 10 seconds. Very few loop the same song for 30 minutes. The bell curve captures this — peak in the middle, thin tails at the extremes.

**When you'll see it in ML:**
- Model prediction errors (a well-trained model makes errors centred near zero)
- Measurement noise in sensor data
- Feature distributions after standardisation
- Default assumption in many statistical tests

In [ ]:
# Simulate streaming session lengths (minutes)
# Mean = 3.2 minutes, std = 0.9 minutes — realistic for music streaming
session_lengths = np.random.normal(loc=3.2, scale=0.9, size=10_000)
session_lengths = session_lengths.clip(0.1, 8)  # clip unrealistic values

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(session_lengths, bins=50, color='teal', edgecolor='white', alpha=0.8, density=True)
ax.set_xlabel('Session length (minutes)', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Simulated streaming session lengths — Normal Distribution', fontsize=13)

# Add mean and ±1σ lines
mu, sigma = np.mean(session_lengths), np.std(session_lengths)
ax.axvline(mu, color='orange', linewidth=2.5, label=f'Mean = {mu:.2f} min')
ax.axvline(mu + sigma, color='red', linewidth=1.5, linestyle='--', label=f'±1σ = {sigma:.2f} min')
ax.axvline(mu - sigma, color='red', linewidth=1.5, linestyle='--')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# The 68-95-99.7 rule: how much data falls within 1, 2, 3 standard deviations
within_1sd = np.sum(np.abs(session_lengths - mu) <= 1*sigma) / len(session_lengths)
within_2sd = np.sum(np.abs(session_lengths - mu) <= 2*sigma) / len(session_lengths)
within_3sd = np.sum(np.abs(session_lengths - mu) <= 3*sigma) / len(session_lengths)

print("The 68-95-99.7 Rule for Normal Distributions:")
print(f"  Within 1 standard deviation: {within_1sd:.1%}  (theory: ~68%)")
print(f"  Within 2 standard deviations: {within_2sd:.1%}  (theory: ~95%)")
print(f"  Within 3 standard deviations: {within_3sd:.1%}  (theory: ~99.7%)")
print()
print(f"So: 95% of users listen between {mu-2*sigma:.1f} and {mu+2*sigma:.1f} minutes")
print(f"    Only 0.3% of sessions are outside {mu-3*sigma:.1f}–{mu+3*sigma:.1f} minutes")

### 💡 The 68-95-99.7 Rule

This is one of the most useful rules in statistics:
- **68%** of data falls within **1 standard deviation** of the mean
- **95%** of data falls within **2 standard deviations** of the mean
- **99.7%** of data falls within **3 standard deviations** of the mean

**Why this matters for ML:** When your model makes an error, you can use this rule to judge whether it's a "normal" error or an outlier worth investigating. An error more than 3 standard deviations from zero suggests something unusual about that data point.

## 🔄 The Central Limit Theorem — "Why the bell curve appears everywhere"

**In plain English:**
> Take **any** distribution. Calculate the average of many random samples from it. Those averages will form a normal distribution — even if the original data isn't bell-shaped at all.

**Why this is mind-blowing:** It means that statistical methods built around the normal distribution work on almost any dataset, as long as your samples are large enough.

**Why ML engineers care:**
- Sample means are approximately normal → you can use z-tests and t-tests on almost any data
- It explains why averaging predictions in ensemble models (like Random Forests) works so well
- It's the mathematical foundation behind why "more data is better"

Let's see it in action by sampling from our **skewed** streaming data.

In [ ]:
# Create a skewed distribution — like user daily listening time
# Most users listen < 1 hour, but some "power users" listen all day
skewed_listening = st.skewnorm.rvs(a=8, loc=0.5, scale=1.2, size=10_000).clip(0, 12)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: the skewed original distribution
axes[0].hist(skewed_listening, bins=50, color='coral', edgecolor='white', alpha=0.8)
axes[0].set_title('Original data: skewed (not a bell curve)', fontsize=12)
axes[0].set_xlabel('Daily listening hours', fontsize=11)
axes[0].axvline(np.mean(skewed_listening), color='navy', linewidth=2, label=f'Mean={np.mean(skewed_listening):.2f}h')
axes[0].legend()

# Right: what happens when we take sample means?
def sample_means(data, sample_size, n_samples):
    return [np.mean(np.random.choice(data, size=sample_size, replace=False))
            for _ in range(n_samples)]

means_n30 = sample_means(skewed_listening, sample_size=30, n_samples=1000)

axes[1].hist(means_n30, bins=40, color='teal', edgecolor='white', alpha=0.8)
axes[1].set_title('Sample means (n=30): becomes a bell curve!', fontsize=12)
axes[1].set_xlabel('Mean daily listening hours (from samples of 30)', fontsize=11)

plt.suptitle('Central Limit Theorem in action', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### ⏸️ Pause and Predict

We just showed CLT with samples of 30. What happens if we use smaller samples (n=5) vs larger samples (n=100)?

**Predict before running the next cell:**
- With n=5: will the distribution of means look more normal or less normal?
- With n=100: will it be narrower or wider than n=30?
- Why?

*Write your prediction here:*

In [ ]:
# Compare how sample size affects the shape of the sampling distribution
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=False)

for ax, n, color in zip(axes, [5, 30, 100], ['coral', 'teal', 'steelblue']):
    means = sample_means(skewed_listening, sample_size=n, n_samples=1000)
    ax.hist(means, bins=35, color=color, edgecolor='white', alpha=0.85)
    ax.set_title(f'Sample size n = {n}\nstd = {np.std(means):.3f}', fontsize=12)
    ax.set_xlabel('Sample mean (hours)', fontsize=10)
    ax.axvline(np.mean(skewed_listening), color='black', linewidth=2, linestyle='--', label='True mean')
    ax.legend(fontsize=9)

plt.suptitle('CLT: larger samples → narrower, more normal distribution of means', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("Key observations:")
print(f"  n=5:   distribution is wide and slightly skewed (not enough samples for CLT to kick in)")
print(f"  n=30:  distribution starts to look normal — CLT working")
print(f"  n=100: distribution is tight and very normal — sample means highly reliable")

### 💡 What do you notice?

- **Small samples (n=5):** The distribution of means is wide and still somewhat skewed. Small samples can mislead you.
- **Large samples (n=30+):** The distribution becomes bell-shaped. This is CLT in action.
- **Very large samples (n=100):** The bell is much narrower — means cluster tightly around the true value.

**The practical rule:** With n ≥ 30, you can usually assume sample means are normally distributed, regardless of the original distribution shape. This is why many statistical rules of thumb use 30 as a minimum sample size.

**Back to our streaming platform:**
> When your data team says "we need at least 30 users per segment to run a reliable analysis", they're applying the Central Limit Theorem — even if they don't say so by name.

## ✅ Section 2 Summary

| Distribution | Shape | Real-world use |
|---|---|---|
| **Uniform** | Flat rectangle | Random assignment, hyperparameter search |
| **Normal** | Bell curve | Errors, measurements, standardised features |

**The Central Limit Theorem (CLT) in one sentence:**
> Sample means follow a normal distribution, regardless of the original data's shape — as long as samples are large enough (n ≥ 30).

**Why CLT matters for ML:**
- It's why averaging works (ensemble methods)
- It justifies using normal-distribution-based statistics on most datasets
- It's why "more data" reduces uncertainty

---
**Up next → Section 3:** How do we use statistics to make decisions? Open `Part_3_probability_statistics_lesson.ipynb`